# Phase 2 Research Gate — one-click runner (Google Colab)

**Automatic:** freezes 2 years of OKX market data + real historical funding rates →
honest backtest (fees, slippage, funding, pessimistic fills) → bias audits → prints
the two reports you paste back to the assistant.

**Usage:**
1. Menu **Runtime → Run all** (or Ctrl/Cmd+F9)
2. Keep this tab open (~30–50 min; the data freeze is the slow part)
3. Scroll to the **last cell**, copy everything, paste to the assistant

If Colab disconnects mid-download: re-run Cell 3 — already-frozen pairs are skipped.

The notebook auto-switches to the arena work branch if `research/` is not on `main` yet.

In [ ]:
# Cell 1 — fetch the repo (auto-fallback to the work branch when PR not merged yet)
import os
REPO = '/content/ML_ANN_Paper_Bot'
if not os.path.exists(REPO):
    os.system(f'git clone -q https://github.com/ah9mohammad-netizen/ML_ANN_Paper_Bot.git {REPO}')
else:
    os.system(f'git -C {REPO} fetch -q origin --prune')
os.chdir(REPO)

if not os.path.exists('research/fetch_data.py'):
    print('⚠️  research/ not found on main — switching to arena work branch')
    os.system('git checkout -q arena/01a027de-ml-ann-paper-bot || '
              'git checkout -q -b arena/01a027de-ml-ann-paper-bot origin/arena/01a027de-ml-ann-paper-bot')

assert os.path.exists('research/fetch_data.py'), \
    'research/ still missing — merge PR #2 on GitHub or re-clone'
print('cwd:', os.getcwd())
print('HEAD:', os.popen('git log --oneline -1').read().strip())
print('research/: ', sorted(os.listdir('research')))

In [ ]:
# Cell 2 — dependencies + OKX connectivity probe (diagnoses network blocks early)
rc = os.system('pip install -q -r requirements-research.txt')
print('pip install rc:', rc)

import requests
ok = False
for host in ['https://aws.okx.com', 'https://www.okx.com', 'https://www.okx.cab', 'https://www.okx.ceo']:
    try:
        t = requests.get(host + '/api/v5/public/time', timeout=10).json()
        if t.get('code') == '0':
            print(f'✅ OKX reachable via {host}')
            ok = True
            break
    except Exception:
        continue
if not ok:
    raise SystemExit('⛔ OKX API is NOT reachable from this Colab runtime. '
                     'Try: Runtime → Disconnect and delete runtime → Run all '
                     '(new VM = new region/IP), or run on your local machine/Railway.')

In [ ]:
# Cell 3 — FREEZE DATA  (slow: 20–40 min; prints one line per pair; resumes if re-run)
DAYS = 730  # covers bear + chop + bull regimes
rc = os.system(f'python -m research.fetch_data --days {DAYS}')
print('fetch return code:', rc)
assert rc == 0, 'data freeze failed'

In [ ]:
# Cell 4 — RUN THE GATE (truth backtest + bias audits; a few minutes)
rc = os.system('python -m research.run_research_gate --frozen')
print('gate return code:', rc)
assert rc == 0, 'gate run failed'

In [ ]:
# Cell 5 — PRINT THE REPORTS  (copy everything below and paste it to the assistant)
from pathlib import Path
out = Path('research/output')
files = sorted(out.glob('REAL_*')) if out.exists() else []
if not files:
    print('⛔ no reports found. research/output contains:',
          sorted(x.name for x in out.glob('*')) if out.exists() else '(missing)')
else:
    print('=' * 30, f'REPORT 1 OF {len(files)}: {files[0].name}', '=' * 30)
    print(files[0].read_text())
    if len(files) > 1:
        print('=' * 30, f'REPORT 2 OF {len(files)}: {files[1].name}', '=' * 30)
        print(files[1].read_text())
    print('=' * 30, 'END — paste everything above this line', '=' * 30)

In [ ]:
# Optional — download trades.csv + manifest as a zip
os.system('cd research && zip -qr /content/research_output.zip output data/frozen/manifest.json')
try:
    from google.colab import files as gfiles
    gfiles.download('/content/research_output.zip')
except Exception as e:
    print('download skipped (not in Colab):', e)